# 01 — Excel Inspection and EDA Readiness

This notebook inspects the raw workbook without cleaning or exporting data. Its main vehicle-volume goal is to map the report layout so a later notebook can parse it safely.

> **Data safety:** Raw previews may contain company data. Before committing, use **Kernel → Restart & Clear Output** and confirm that every output and execution count is empty.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.load_excel import find_excel_file, inspect_sheet, list_sheet_names

## 1. Locate and inspect the workbook

Sheets are read first with `header=None`, preserving report titles, merged-cell gaps, and multi-row headers.

In [ ]:
workbook_path = find_excel_file(PROJECT_ROOT / "data" / "raw")
sheet_names = list_sheet_names(workbook_path)
inspections = {
    sheet_name: inspect_sheet(workbook_path, sheet_name)
    for sheet_name in sheet_names
}

print(f"Workbook: {workbook_path.name}")
print("Sheets:")
for sheet_name in sheet_names:
    result = inspections[sheet_name]
    print(f"- {sheet_name}: shape={result['shape']}, format={result['inferred_format']}")

In [ ]:
required_keys = {
    "sheet_name", "shape", "raw_preview", "raw_column_names",
    "header_column_names", "missing_summary", "inferred_format",
    "useful_fields", "cleaning_notes", "report_structure", "pio_profile",
}
for sheet_name, result in inspections.items():
    missing_keys = required_keys - result.keys()
    if missing_keys:
        raise KeyError(f"{sheet_name} inspection is missing: {sorted(missing_keys)}")
print("Inspection return-schema check passed.")

## 2. Raw sheet previews

These previews are for visual inspection only and must be cleared before committing.

In [ ]:
for sheet_name, result in inspections.items():
    display(Markdown(f"### {sheet_name}"))
    print(f"Shape: {result['shape']}")
    print(f"Inferred format: {result['inferred_format']}")
    display(result["raw_preview"])
    display(Markdown("**Missing-value summary**"))
    display(result["missing_summary"])
    display(Markdown("**Inspection / cleaning considerations**"))
    for note in result["cleaning_notes"]:
        print(f"- {note}")

### PIO field interpretation note

- `PIS_SERI` appears to align closely with the vehicle model code in `Vehicle_Wholesale_Data`. It is useful for validating mappings and variant differences, but should not be used as the only matching key.
- `PIS_PNO` is the stable accessory key; `Model` is the vehicle model name.
- The `SumOf...` fields suggest that `PIO_Sales_Data` may already be aggregated, so its row-level grain needs Mobis confirmation.
- Observed `PIS_CMP_KND` values appear to be H and K. H may include Hyundai and Genesis because Genesis models appear without a separate G code.
- Later matching to HMA/GMA/KUS report groups should combine brand, model name, month, and model-code evidence rather than relying only on `PIS_CMP_KND` or `PIS_SERI`.

## 3. `Vehicle_Wholesale_Data` report-layout map

Row numbers below are Excel-style (starting at 1). Section ranges are estimates: each section starts at its detected label and ends immediately before the next section, or at the last used row.

In [ ]:
vehicle_result = inspections.get("Vehicle_Wholesale_Data")
if vehicle_result is None:
    raise KeyError("Vehicle_Wholesale_Data was not found in the workbook.")

structure = vehicle_result["report_structure"]
if structure is None:
    raise ValueError("Vehicle report structure was not generated.")

layout_rows = []
for section in structure["sections"]:
    brand_blocks = ", ".join(
        f"{'/'.join(item['matched_values']).upper()} (row {item['row_number']})"
        for item in section["brand_blocks"]
    ) or "None detected"
    subtotal_rows = ", ".join(
        f"{'/'.join(item['matched_values'])} (row {item['row_number']})"
        for item in section["subtotal_rows"]
    ) or "None detected"
    layout_rows.append({
        "section": section["section"],
        "row range": section["row_range"],
        "brand blocks": brand_blocks,
        "subtotal rows": subtotal_rows,
        "notes": "Estimated boundary; parse separately and visually confirm.",
    })

display(pd.DataFrame(layout_rows))

### Candidate year/month columns

Year labels are forward-filled across repeated month columns for inspection only. This does not reshape or clean the report.

In [ ]:
year_month_columns = structure["year_month_column_candidates"]
if year_month_columns:
    display(pd.DataFrame(year_month_columns))
else:
    print("No year/month column candidates detected.")

print(f"Wide year/month layout detected: {structure['has_wide_year_month_columns']}")
print("Possible year header rows:", structure["possible_year_header_rows"])
print("Possible month header rows:", structure["possible_month_header_rows"])

## 4. Cleaning-readiness notes

- Wholesale and Fleet should be parsed separately.
- Subtotal rows should likely be excluded from model-level cleaning to avoid double counting.
- Year/month headers must be reconstructed before wide-to-long reshaping.
- Detected row ranges and headers are candidates and should be visually confirmed before cleaning logic is implemented.
- `header=0` names such as `Unnamed: 0` are report-layout artifacts, not cleaned column names.

**Stopping point:** this notebook creates no cleaned tables, CSV files, processed workbooks, forecasts, or dashboards.

> **Before committing:** clear all outputs and execution counts. The notebook must contain code and markdown only.